# 02 — Feature Engineering

**Goal:** Build a unified feature schema across sorting and searching datasets,
suitable for both the regression (execution-time) and classification
(best-algorithm) models, without leaking the target into the inputs.

In [1]:
import pandas as pd
import numpy as np

sorting_df = pd.read_csv("../data/processed/sorting_clean.csv")
searching_df = pd.read_csv("../data/processed/searching_clean.csv")

print(sorting_df.shape, searching_df.shape)

(34992, 11) (29788, 10)


In [3]:
sorting = sorting_df.copy()
sorting["domain"] = "sorting"
sorting = sorting.rename(columns={
    "data_type": "input_condition",
    "swaps": "swaps_or_shifts",
})

searching = searching_df.copy()
searching["domain"] = "searching"
searching = searching.rename(columns={
    "case_type": "input_condition",
})

print(sorting.columns.tolist())
print(searching.columns.tolist())

['algorithm', 'input_size', 'input_condition', 'trial_id', 'comparisons', 'swaps_or_shifts', 'time_taken', 'memory_used', 'presortedness_score', 'duplicate_ratio', 'value_range', 'domain']
['algorithm', 'input_size', 'input_condition', 'trial_id', 'probes', 'time_taken', 'memory_used', 'found', 'load_factor', 'collision_count', 'domain']


In [4]:
unified_df = pd.concat([sorting, searching], ignore_index=True, sort=False)

print(unified_df.shape)
unified_df.head()

(64780, 16)


,algorithm,input_size,input_condition,trial_id,comparisons,swaps_or_shifts,time_taken,memory_used,presortedness_score,duplicate_ratio,value_range,domain,probes,found,load_factor,collision_count
0,bubble_sort,10,random,1,44.0,30.0,0.000039,416,0.3333,0.0,88143.0,sorting,NaN,NaN,NaN,NaN
1,selection_sort,10,random,1,45.0,8.0,0.000025,416,0.3333,0.0,88143.0,sorting,NaN,NaN,NaN,NaN
2,insertion_sort,10,random,1,36.0,30.0,0.000026,416,0.3333,0.0,88143.0,sorting,NaN,NaN,NaN,NaN
3,merge_sort,10,random,1,22.0,34.0,0.000074,416,0.3333,0.0,88143.0,sorting,NaN,NaN,NaN,NaN
4,quick_sort,10,random,1,29.0,24.0,0.000072,416,0.3333,0.0,88143.0,sorting,NaN,NaN,NaN,NaN


In [6]:
print(unified_df.isnull().sum())
print(unified_df.groupby("domain")[["comparisons", "swaps_or_shifts", "presortedness_score",
                                      "duplicate_ratio", "value_range", "probes", "found",
                                      "load_factor", "collision_count"]].apply(lambda g: g.notna().sum()))

algorithm                  0
input_size                 0
input_condition            0
trial_id                   0
comparisons            29788
swaps_or_shifts        29788
time_taken                 0
memory_used                0
presortedness_score    29788
duplicate_ratio        29788
value_range            29788
domain                     0
probes                 34992
found                  34992
load_factor            59780
collision_count        59780
dtype: int64
           comparisons  swaps_or_shifts  presortedness_score  duplicate_ratio  \
domain                                                                          
searching            0                0                    0                0   
sorting          34992            34992                34992            34992   

           value_range  probes  found  load_factor  collision_count  
domain                                                               
searching            0   29788  29788         5000        

In [7]:
unified_df["is_hashing"] = (unified_df["algorithm"] == "hashing_search").astype(int)

print(unified_df["is_hashing"].value_counts())
print(unified_df[unified_df["is_hashing"] == 1]["load_factor"].describe())

is_hashing
0    59780
1     5000
Name: count, dtype: int64
count    5000.0
mean        2.0
std         0.0
min         2.0
25%         2.0
50%         2.0
75%         2.0
max         2.0
Name: load_factor, dtype: float64


## Unified Schema — Notes

- Merged sorting (34,992 rows) + searching (29,788 rows) = 64,780 rows total.
- Domain-specific columns (`comparisons`, `swaps_or_shifts`, `presortedness_score`,
  `duplicate_ratio`, `value_range` for sorting; `probes`, `found`, `load_factor`,
  `collision_count` for searching) are NaN on the "wrong" domain by construction —
  this is structural (not applicable), not missing data, and will NOT be imputed
  with mean/median.
- `load_factor` is constant (2.0) for all hashing rows — zero variance, carries
  no predictive signal. Kept in the unified table for documentation but will be
  excluded from actual model input features.
- `is_hashing` flag added as an explicit indicator, since `load_factor`/
  `collision_count` are only meaningful for `hashing_search`.

In [8]:
# Columns common to both domains (used as base features for regression)
common_categorical = ["domain", "algorithm", "input_condition"]
common_numerical = ["input_size"]

# Domain-specific numerical features
sorting_only_numerical = ["comparisons", "swaps_or_shifts", "presortedness_score",
                           "duplicate_ratio", "value_range"]
searching_only_numerical = ["probes", "collision_count"]  # load_factor excluded (constant)

# Target for regression
regression_target = "time_taken"

print("Common categorical:", common_categorical)
print("Common numerical:", common_numerical)
print("Sorting-only numerical:", sorting_only_numerical)
print("Searching-only numerical:", searching_only_numerical)

Common categorical: ['domain', 'algorithm', 'input_condition']
Common numerical: ['input_size']
Sorting-only numerical: ['comparisons', 'swaps_or_shifts', 'presortedness_score', 'duplicate_ratio', 'value_range']
Searching-only numerical: ['probes', 'collision_count']


In [9]:
model_df = unified_df.copy()

structural_cols = sorting_only_numerical + searching_only_numerical
model_df[structural_cols] = model_df[structural_cols].fillna(0)

# Confirm no NaNs remain in our feature columns
feature_cols = common_categorical + common_numerical + structural_cols
print(model_df[feature_cols].isnull().sum())

domain                 0
algorithm              0
input_condition        0
input_size             0
comparisons            0
swaps_or_shifts        0
presortedness_score    0
duplicate_ratio        0
value_range            0
probes                 0
collision_count        0
dtype: int64


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numerical_features = common_numerical + structural_cols
categorical_features = common_categorical

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['input_size', 'comparisons', 'swaps_or_shifts', 'presortedness_score', 'duplicate_ratio', 'value_range', 'probes', 'collision_count']
Categorical features: ['domain', 'algorithm', 'input_condition']


In [11]:
X = model_df[numerical_features + categorical_features]
y = model_df[regression_target]

print(X.shape, y.shape)
X.head()

(64780, 11) (64780,)


,input_size,comparisons,swaps_or_shifts,presortedness_score,duplicate_ratio,value_range,probes,collision_count,domain,algorithm,input_condition
0,10,44.0,30.0,0.3333,0.0,88143.0,0.0,0.0,sorting,bubble_sort,random
1,10,45.0,8.0,0.3333,0.0,88143.0,0.0,0.0,sorting,selection_sort,random
2,10,36.0,30.0,0.3333,0.0,88143.0,0.0,0.0,sorting,insertion_sort,random
3,10,22.0,34.0,0.3333,0.0,88143.0,0.0,0.0,sorting,merge_sort,random
4,10,29.0,24.0,0.3333,0.0,88143.0,0.0,0.0,sorting,quick_sort,random


In [12]:
from sklearn.model_selection import GroupShuffleSplit

model_df["config_id"] = (
    model_df["domain"] + "_" +
    model_df["algorithm"] + "_" +
    model_df["input_size"].astype(str) + "_" +
    model_df["input_condition"]
)

print("Unique configs:", model_df["config_id"].nunique())
print("Rows per config (should mostly be 5):")
print(model_df["config_id"].value_counts().describe())

Unique configs: 13000
Rows per config (should mostly be 5):
count    13000.000000
mean         4.983077
std          0.140411
min          3.000000
25%          5.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: count, dtype: float64


In [13]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=model_df["config_id"]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Sanity check: no config appears in both train and test
train_configs = set(model_df.iloc[train_idx]["config_id"])
test_configs = set(model_df.iloc[test_idx]["config_id"])
print("Overlap between train/test configs:", len(train_configs & test_configs))

Train shape: (51832, 11)
Test shape: (12948, 11)
Overlap between train/test configs: 0


## Train/Test Split Strategy

A plain random `train_test_split` would risk leakage: each (domain, algorithm,
input_size, input_condition) configuration has ~5 repeated trials with nearly
identical execution times (same config, small timing noise). Randomly splitting
at the row level could place some trials of a config in training and others in
testing, letting the model partially memorize that config's typical timing
rather than genuinely generalize to unseen configurations.

**Solution:** `GroupShuffleSplit` groups all trials of a configuration together
using a composite `config_id` (domain + algorithm + input_size + input_condition),
guaranteeing every trial of a given config lands entirely in train or entirely
in test. Verified: zero configuration overlap between the resulting train/test
sets.

In [15]:
import os

os.makedirs("../data/processed", exist_ok=True)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

model_df.to_csv("../data/processed/unified_dataset.csv", index=False)

print("Saved train/test splits and unified dataset.")

Saved train/test splits and unified dataset.


In [16]:
check = pd.read_csv("../data/processed/X_train.csv")
print(check.shape)
check.head()

(51832, 11)


,input_size,comparisons,swaps_or_shifts,presortedness_score,duplicate_ratio,value_range,probes,collision_count,domain,algorithm,input_condition
0,10,45.0,8.0,0.3333,0.0,88143.0,0.0,0.0,sorting,selection_sort,random
1,10,36.0,30.0,0.3333,0.0,88143.0,0.0,0.0,sorting,insertion_sort,random
2,10,22.0,34.0,0.3333,0.0,88143.0,0.0,0.0,sorting,merge_sort,random
3,10,29.0,24.0,0.3333,0.0,88143.0,0.0,0.0,sorting,quick_sort,random
4,10,37.0,24.0,0.3333,0.0,88143.0,0.0,0.0,sorting,heap_sort,random


In [17]:
check_y = pd.read_csv("../data/processed/y_train.csv")
print(check_y.shape)
check_y.head()

(51832, 1)


,time_taken
0,0.000025
1,0.000026
2,0.000074
3,0.000072
4,0.000057
